# 第5周 Day7 · 第五周总复习——推理能力全景概念实验

> 本 notebook 是《第5周-Day7-第五周总复习.md》的可执行配套实验。核心问题：**CoT、ToT、自洽性投票、R1 式推理各自"买到"多少正确率，付出多少成本？**
>
> 实验仅用 numpy / 标准库 / matplotlib 做教学级蒙特卡洛模拟（不含真实模型调用），目标是把"技术选型能力"变成可计算的曲线：
>
> 1. 多步任务模拟器：为什么单步可靠度的微小提升会被**指数放大**
> 2. 自洽性投票：多数票正确率随采样次数 k 的提升 vs 线性增长的成本
> 3. ToT 分支搜索：候选数 b 的收益递减 + 评估器瓶颈
> 4. Pareto 前沿：各策略"效果-成本"对比与选型方法论

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("实验环境就绪。本周技术地图（来自 md）：")
print(f"{'技术':<10s}{'核心问题':<22s}{'主要代价'}")
_map = [
    ("CoT",      "如何避免跳步",       "输出变长"),
    ("ToT",      "如何避免一条路走错", "分支计算多"),
    ("自洽性",   "如何交叉验证",       "多次采样"),
    ("R1式推理", "如何把深思考训练进模型", "延迟与Token高"),
    ("RAG",      "如何获得新事实",     "依赖检索质量"),
    ("Agent",    "如何把推理变行动",   "权限与安全复杂"),
]
for row in _map:
    print(f"{row[0]:<10s}{row[1]:<22s}{row[2]}")

## 实验 1：多步任务模拟器——单步提升被指数放大

假设一个任务需要连续 5 步推导，**全部正确**才能得到正确答案（数学题的典型形态）。不同策略的差异，先简化为"单步可靠度"的差异（教学假设，非真实榜单）：

| 策略 | 单步可靠度 | 相对成本 |
|---|---|---|
| 零样本 | 0.78 | 1x |
| CoT | 0.90 | 2.2x |
| ToT | 0.94 | 4.5x |
| R1式推理 | 0.96 | 5.5x |

In [ ]:
strategies = {
    "零样本":   {"step_acc": 0.78, "cost": 1.0},
    "CoT":      {"step_acc": 0.90, "cost": 2.2},
    "ToT":      {"step_acc": 0.94, "cost": 4.5},
    "R1式推理": {"step_acc": 0.96, "cost": 5.5},
}

N_TASKS, N_STEPS = 4000, 5

def simulate_accuracy(step_acc, n_tasks=N_TASKS, n_steps=N_STEPS, seed=0):
    """蒙特卡洛：每步以 step_acc 概率不出错，全链路正确才算任务正确。"""
    r = np.random.default_rng(seed)
    correct = r.random((n_tasks, n_steps)) < step_acc
    return correct.all(axis=1).mean()

results = {}
for name, s in strategies.items():
    acc = simulate_accuracy(s["step_acc"], seed=5)
    results[name] = {"acc": acc, "cost": s["cost"]}
    theo = s["step_acc"] ** N_STEPS
    print(f"{name:<8s} 单步={s['step_acc']:.2f} → 5步全对 ≈ {acc:.3f}（理论值 {theo:.3f}）")

print()
print("观察：0.78 → 0.96 的单步提升，在 5 步任务上被放大为 "
      f"{results['零样本']['acc']:.3f} → {results['R1式推理']['acc']:.3f}。")
print("这就是'跳步'代价：任何一步出错，后面推导再漂亮也是错的（CoT 的价值来源）。")

## 实验 2：自洽性（Self-consistency）——多次采样 + 多数投票

自洽性的做法：同一问题独立采样 k 次，对最终答案做**多数投票**。教学模型假设：
- 每次解答以概率 `p`（取实验 1 中 CoT 的任务级正确率）正确；
- 错误答案各不相同，因此**正确票过半**即得到正确答案；
- 成本随 k **线性**增长——正确率提升却有上限，存在性价比拐点。

In [ ]:
p_cot = results["CoT"]["acc"]          # CoT 单次解答的任务级正确率
print(f"CoT 单次解答正确率 p = {p_cot:.3f}\n")

def majority_vote_acc(p, k, n_tasks=N_TASKS, seed=7):
    """k 次独立采样后多数投票：正确票 > k/2 视为任务正确。"""
    r = np.random.default_rng(seed)
    n_correct = (r.random((n_tasks, k)) < p).sum(axis=1)
    return (n_correct > k / 2).mean()

ks = [1, 3, 5, 7, 9, 11, 15]
sc_acc = [majority_vote_acc(p_cot, k) for k in ks]
sc_cost = [k * strategies["CoT"]["cost"] for k in ks]

print(f"{'k':>3s}{'投票正确率':>12s}{'相对成本':>12s}{'边际增益/单位成本':>18s}")
for i, k in enumerate(ks):
    if i == 0:
        print(f"{k:>3d}{sc_acc[i]:>11.3f}{sc_cost[i]:>11.1f}x{'基线':>16s}")
        continue
    gain = sc_acc[i] - sc_acc[i - 1]
    dcost = k - ks[i - 1]
    print(f"{k:>3d}{sc_acc[i]:>11.3f}{sc_cost[i]:>11.1f}x{gain / dcost:>17.4f}")

print("\n观察：k 从 1→5 正确率提升明显，之后边际收益快速衰减；")
print("而成本严格线性增长——'多算几次取多数'存在性价比拐点（高价值判断才值得用）。")

## 实验 3：ToT（Tree-of-Thought）——分支搜索 + 评估器瓶颈

ToT 对每个问题生成 `b` 条**候选思路**再择优。教学模型：
- 每条候选以概率 `p`（CoT 任务级正确率）正确，至少有一条正确即有机会解出；
- 但需要**评估器**从候选中挑出正确的那条，评估器只有 `p_eval` 的辨别力；
- 成本 ≈ b 条候选 + 评估开销。

关键洞察：ToT 的上限由 `min(覆盖度, 评估器)` 共同决定——评估器不灵，分支再多也白搭。

In [ ]:
P_EVAL = 0.75   # 评估器从正确候选中挑出最优的概率（教学假设）

def tot_accuracy(p, b, p_eval=P_EVAL, n_tasks=N_TASKS, seed=11):
    """b 分支候选 + 评估器择优：需要(存在正确候选)且(评估器挑中)。"""
    r = np.random.default_rng(seed)
    has_correct = (r.random((n_tasks, b)) < p).any(axis=1)
    picked = r.random(n_tasks) < p_eval
    return (has_correct & picked).mean()

breaths = [1, 2, 3, 4, 6]
tot_acc = [tot_accuracy(p_cot, b) for b in breaths]
tot_cost = [b * strategies["CoT"]["cost"] * 1.3 for b in breaths]  # 1.3x 为评估开销

print(f"{'b':>3s}{'ToT正确率':>12s}{'存在正确候选':>14s}{'评估器瓶颈后':>14s}{'相对成本':>10s}")
for b, acc, c in zip(breaths, tot_acc, tot_cost):
    cover = 1 - (1 - p_cot) ** b
    ceiling = cover * P_EVAL
    print(f"{b:>3d}{acc:>11.3f}{cover:>13.3f}{ceiling:>13.3f}{c:>9.1f}x")

print("\n观察：b=1→3 提升明显，b 再大收益递减（覆盖度饱和）；")
print(f"评估器 p_eval={P_EVAL} 把上限压在覆盖度×0.75——先把'判别哪条思路对'做好，再谈多分支。")

## 实验 4：策略 Pareto 前沿——效果-成本权衡

把五种方案放到同一张"正确率-成本"图上（RAG+Agent 的教学估计值一并加入）：
- **Pareto 前沿** = 不存在另一个点同时更便宜且更准的策略集合；
- 前沿会随假设变化（真实系统中 R1 延迟高、RAG+Agent 工程复杂），**方法论比结论重要**。

In [ ]:
def pareto_front(pts):
    """返回非支配点下标：没有别的点同时满足 成本≤ 且 正确率≥ 且至少一项严格更优。"""
    front = []
    for i, (_, c1, a1) in enumerate(pts):
        dominated = any(
            c2 <= c1 and a2 >= a1 and (c2 < c1 or a2 > a1)
            for (_, c2, a2) in pts
        )
        if not dominated:
            front.append(i)
    return sorted(front, key=lambda i: pts[i][1])

candidates = [
    ("零样本",          strategies["零样本"]["cost"],  results["零样本"]["acc"]),
    ("CoT",             strategies["CoT"]["cost"],     results["CoT"]["acc"]),
    ("CoT+自洽(k=5)",   sc_cost[2],                    sc_acc[2]),
    ("ToT(b=3)",        tot_cost[2],                   tot_acc[2]),
    ("R1式推理",        strategies["R1式推理"]["cost"], results["R1式推理"]["acc"]),
    ("RAG+Agent",       4.0,                           0.87),   # 教学估计：补事实来源后的示意值
]

front_idx = pareto_front(candidates)
front = [candidates[i] for i in front_idx]

print("全部候选（成本↑ / 正确率↑）：")
for n, c, a in sorted(candidates, key=lambda x: x[1]):
    tag = "  ← Pareto" if any(n == fn for fn, _, _ in front) else ""
    print(f"  {n:<14s} 成本≈{c:5.1f}x  正确率≈{a:.3f}{tag}")
print("\n注意：RAG+Agent 的 0.87 不是'推理更强'，而是补上了事实来源——")
print("检索正确但推理错误 / 推理正确但事实过期，是两类不同的失败，需要分别防范。")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8))

ax1.plot(ks, sc_acc, "o-", color="#d96d73", label="投票正确率")
ax1.set_xlabel("采样次数 k")
ax1.set_ylabel("任务正确率", color="#d96d73")
ax1.grid(alpha=0.3)
ax1b = ax1.twinx()
ax1b.bar(ks, sc_cost, alpha=0.3, color="#4c9f9a", label="相对成本")
ax1b.set_ylabel("相对成本", color="#4c9f9a")
ax1.set_title("自洽性投票：正确率提升 vs 成本线性增长")

ax2.scatter([c for _, c, _ in candidates], [a for _, _, a in candidates],
            s=90, color="#555", zorder=3)
for n, c, a in candidates:
    ax2.annotate(n, (c, a), textcoords="offset points", xytext=(8, 5), fontsize=10)
ax2.step([c for _, c, _ in front], [a for _, _, a in front],
         where="post", alpha=0.55, color="#d96d73", linewidth=2, label="Pareto 前沿")
ax2.set_xlabel("相对成本")
ax2.set_ylabel("任务正确率")
ax2.set_title("推理策略效果-成本 Pareto 对比（教学模拟）")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 复盘要点（对应 md 决策树）

1. **选型看四问**：要新事实？→ RAG/工具；≥2 步推导？→ CoT/推理模型；多可行方案？→ ToT/候选比较；高风险外部动作？→ 验证 + 人工确认。
2. **指数放大**是双刃剑：单步可靠度提升在多步任务上收益巨大，同样任何一步疏漏也会被放大——所以 CoT 强调"每步可检查"。
3. **自洽性与 ToT 都受边际递减约束**：先提高单次质量与评估器辨别力，再增加采样/分支数。
4. **Pareto 前沿是动态的**：改变延迟要求、token 单价、任务难度分布，最优选择就会移动——做技术选型时，画自己的前沿，而不是背结论。

> 一页纸复盘法：事实 / 推理 / 验证 / 行动 四栏定位问题——事实错修检索，结论错修推理，动作错修权限与流程。